# 02 — Features por Bloque + Materia Individual (V11)

**Anterior**: `01_assembly.ipynb`  |  **Siguiente**: `03_eda.ipynb`
Lee `dataset_v11.csv`, añade features de notas 1EV en DOS niveles y lo sobreescribe.

## Dos niveles de granularidad (para explicabilidad)
1. **Bloques temáticos** (densos, comparables entre niveles): `__LING / __STEM / __SOC / __PERS`
   con métricas nota_media / nota_min / nota_max / pct_aprobado / pct_asignaturas.
2. **Materia individual canónica** (`mat__Matematicas`, `mat__Lengua_Catalana`…):
   variantes de nombre unificadas con `materias_map.canonical_materia`. Permiten ver
   qué asignatura concreta pesa en el riesgo (importancia de features en NB07/08).

El mapeo vive en `src/utils/materias_map.py` (reutilizable, 0% sin clasificar).

In [1]:
import sys
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
pd.set_option('display.max_columns', 80); pd.set_option('display.float_format','{:.3f}'.format)
sns.set_theme(style='whitegrid', palette='muted'); plt.rcParams['figure.dpi']=110

_cwd = Path('.').resolve()
BASE_DIR = next((p for p in [_cwd,_cwd.parent,_cwd.parent.parent] if (p/'data').exists()), _cwd.parent)
sys.path.append(str(BASE_DIR))
from src.utils.materias_map import asignar_bloque, canonical_materia, BLOQUES

P_NOTAS = BASE_DIR/'data'/'exports'/'querys_v11'/'Q_01_v11_Final.csv'
P_DATASET = BASE_DIR/'data'/'dataset_v11.csv'
KEY = ['GuidAlumno','Ejercicio']

In [2]:
# Cargar notas y dataset base
COL = ['GuidAlumno','Ejercicio','NivEstudio','NivCurso','IsRepetidor',
       'CodigoAsignatura','NombreAsignatura','NotaNumerico','IsAprobado','n_raw_rows']
notas = pd.read_csv(P_NOTAS, sep=';', header=None, names=COL, decimal=',',
                    dtype={'GuidAlumno':str,'Ejercicio':int}, na_values=['NULL',''])
notas['NotaNumerico']=pd.to_numeric(notas['NotaNumerico'],errors='coerce')
notas['IsAprobado']=pd.to_numeric(notas['IsAprobado'],errors='coerce')

df_base = pd.read_csv(P_DATASET)
cat_order=['buen_alumno','en_riesgo','con_dificultades']
df_base['categoria_target']=pd.Categorical(df_base['categoria_target'],categories=cat_order,ordered=True)

# Alinear notas con el cohorte filtrado del dataset
notas = notas.merge(df_base[KEY].drop_duplicates(), on=KEY, how='inner')
notas_v = notas.dropna(subset=['NotaNumerico']).copy()
notas_v['bloque'] = notas_v['NombreAsignatura'].apply(asignar_bloque)
notas_v['materia_canon'] = notas_v['NombreAsignatura'].apply(canonical_materia)
print(f'Notas alineadas: {len(notas_v)}  |  alumnos-anio: {notas_v[KEY].drop_duplicates().shape[0]}')
print('\nPor bloque:'); print(notas_v['bloque'].value_counts())
print(f"\nMateria canonica asignada: {notas_v['materia_canon'].notna().sum()}/{len(notas_v)}")
print('Top materias canonicas:'); print(notas_v['materia_canon'].value_counts().head(20))

Notas alineadas: 8812  |  alumnos-anio: 1205

Por bloque:
bloque
LING    3201
SOC     2692
STEM    2512
PERS     361
ACT       46
Name: count, dtype: int64

Materia canonica asignada: 8292/8812
Top materias canonicas:
materia_canon
Lengua_Castellana      1133
Matematicas            1098
Ciencias               1079
Educacion_Fisica       1028
Educacion_Artistica    1024
Lengua_Catalana        1016
Lengua_Extranjera       918
Sociales                598
Tecnologia              299
Religion_Valores         99
Name: count, dtype: int64


## 1. Features por bloque temático (LING / STEM / SOC / PERS)

In [3]:
acad = notas_v[notas_v['bloque']!='ACT'].copy()
feats_comp = acad.groupby(KEY+['bloque']).agg(
    nota_media=('NotaNumerico','mean'), nota_min=('NotaNumerico','min'),
    nota_max=('NotaNumerico','max'), n_asig=('NombreAsignatura','nunique'),
    pct_aprobado=('IsAprobado','mean')).reset_index()
feats_comp = feats_comp.merge(df_base[KEY+['n_asignaturas_1ev']], on=KEY, how='left')
feats_comp['pct_asignaturas'] = feats_comp['n_asig']/feats_comp['n_asignaturas_1ev'].replace(0,np.nan)

blk = feats_comp.pivot_table(index=KEY, columns='bloque',
        values=['nota_media','nota_min','nota_max','pct_aprobado','pct_asignaturas'])
blk.columns=[f'{m}__{b}' for m,b in blk.columns]; blk=blk.reset_index()
print(f'Features por bloque: {blk.shape[1]-2} columnas')
cobertura = {c: f'{df_base[KEY].merge(blk[KEY+[c]],on=KEY,how="left")[c].notna().mean()*100:.0f}%'
             for c in blk.columns if '__' in c}
print('Cobertura por bloque (nota_media):')
for b in BLOQUES:
    c=f'nota_media__{b}'
    if c in blk.columns: print(f'  {b}: {cobertura.get(c)}')

Features por bloque: 20 columnas
Cobertura por bloque (nota_media):
  LING: 99%
  STEM: 92%
  SOC: 91%
  PERS: 25%


## 2. Features por materia individual canónica (`mat__X`)

In [4]:
ind = acad.dropna(subset=['materia_canon']).copy()
mat = ind.groupby(KEY+['materia_canon'])['NotaNumerico'].mean().reset_index()
mat = mat.pivot_table(index=KEY, columns='materia_canon', values='NotaNumerico')
mat.columns=[f'mat__{c}' for c in mat.columns]; mat=mat.reset_index()
print(f'Materias individuales: {mat.shape[1]-2} columnas')
cov = (mat.drop(columns=KEY).notna().sum().sort_values(ascending=False))
print('Cobertura por materia (nº alumnos con nota):')
for m,n in cov.items(): print(f'  {m:<28} {n:>4}  ({100*n/len(df_base):.0f}%)')

Materias individuales: 10 columnas
Cobertura por materia (nº alumnos con nota):
  mat__Lengua_Castellana       1133  (94%)
  mat__Matematicas             1085  (90%)
  mat__Educacion_Fisica        1028  (85%)
  mat__Lengua_Catalana         1016  (84%)
  mat__Ciencias                 986  (82%)
  mat__Educacion_Artistica      900  (75%)
  mat__Lengua_Extranjera        888  (74%)
  mat__Sociales                 446  (37%)
  mat__Tecnologia               274  (23%)
  mat__Religion_Valores          99  (8%)


## 3. Merge + guardar dataset enriquecido

In [5]:
# Quitar columnas de features previas si se re-ejecuta
prev=[c for c in df_base.columns if c.startswith('mat__') or '__' in c]
df = df_base.drop(columns=prev, errors='ignore').merge(blk,on=KEY,how='left').merge(mat,on=KEY,how='left')
if 'target_num' not in df.columns: df['target_num']=df['categoria_target'].cat.codes
df.to_csv(P_DATASET, index=False)

cols_blk=[c for c in df.columns if '__' in c and not c.startswith('mat__')]
cols_mat=[c for c in df.columns if c.startswith('mat__')]
print(f'Dataset enriquecido: {df.shape}')
print(f'  Features bloque:    {len(cols_blk)}')
print(f'  Features materia:   {len(cols_mat)}')
print(f'Guardado: {P_DATASET}')

Dataset enriquecido: (1205, 59)
  Features bloque:    20
  Features materia:   10
Guardado: C:\Users\emili\OneDrive\Escritorio\TFM\data\dataset_v11.csv
